# **LazImpa - Transformer Edition**

*Based on original work by Mathieu Rita*

**Extended with Transformer architecture support**

_______________
**INTRODUCTION**

This notebook shows how to run the **Transformer-based** experiments from the LazImpa project. This extends the original LSTM-based experiments with Transformer architectures for both sender and receiver agents.

The notebook is organized in 3 sections:

- I- Clone LazImpa repository and set the environment
- II- Train LazImpa with Transformers
- III- Analyse results and compare with LSTM

_________________

## I - Clone LazImpa repository and set the environment

In [ ]:
# Clone repository (skip if running locally)
!git clone https://github.com/MathieuRita/Lazimpa.git
!mv "./Lazimpa/egg" "./egg"
!mv "./Lazimpa/run_experiments.py" "./run_experiments.py"
!mv "./Lazimpa/experiments_config.json" "./experiments_config.json"

In [ ]:
# Create directories for Transformer experiments
!mkdir -p transformer_baseline/{accuracy,messages,sender,receiver,logs}
!mkdir -p transformer_lazimpa/{accuracy,messages,sender,receiver,logs}
!mkdir -p analysis_transformer

In [ ]:
# Useful functions (for analysis)
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

def load_message(expe):
    """
    Load messages stored during training procedure.
    Return numpy array with all the messages
    """
    messages = np.load(expe, allow_pickle=True)
    return messages

def get_message_lengths(messages):
    """Get length of each message (until EOS token 0)."""
    lengths = []
    for msg in messages:
        # Find first 0 (EOS) or use full length
        if isinstance(msg, np.ndarray):
            eos_pos = np.where(msg == 0)[0]
            if len(eos_pos) > 0:
                lengths.append(eos_pos[0] + 1)
            else:
                lengths.append(len(msg))
        else:
            lengths.append(len(msg))
    return lengths

def plot_length_distribution(messages, title="Message Length Distribution"):
    """Plot message length distribution."""
    lengths = get_message_lengths(messages)
    plt.figure(figsize=(10, 5))
    plt.plot(lengths)
    plt.xlabel("Input (ranked by frequency)")
    plt.ylabel("Message Length")
    plt.title(title)
    plt.grid(alpha=0.3)
    plt.show()
    return lengths

## II - Train LazImpa with Transformers

### II.1 - Transformer Baseline (Standard Agents)

Standard Transformer agents without lazy speaker or impatient listener.

In [ ]:
# TRANSFORMER BASELINE - Standard Agents
# For quick testing, use n_features=100, n_epochs=100
# For full replication, use n_features=1000, n_epochs=500

!python -m egg.zoo.channel.train \
    --dir_save=transformer_baseline \
    --n_features=100 \
    --vocab_size=40 \
    --max_len=30 \
    --batch_size=32 \
    --batches_per_epoch=1000 \
    --n_epochs=100 \
    --lr=0.001 \
    --probs="powerlaw" \
    --random_seed=42 \
    --sender_cell="transformer" \
    --receiver_cell="transformer" \
    --sender_hidden=512 \
    --receiver_hidden=512 \
    --sender_embedding=256 \
    --receiver_embedding=256 \
    --sender_num_layers=2 \
    --receiver_num_layers=2 \
    --sender_num_heads=8 \
    --receiver_num_heads=8 \
    --causal_sender \
    --causal_receiver \
    --sender_generate_style="in-place" \
    --sender_entropy_coeff=0.1 \
    --receiver_entropy_coeff=0.1 \
    --impatient=0 \
    --reg=0 \
    --length_cost=0.0 \
    --early_stopping_thr=0.9999

### II.2 - Transformer LazImpa (Lazy Speaker + Impatient Listener)

Transformer agents with:
- **Lazy Speaker** (`reg=1`): Regularization to encourage shorter messages
- **Impatient Listener** (`impatient=1`): Listener makes predictions at each position

In [ ]:
# TRANSFORMER LAZIMPA - Lazy + Impatient Agents
# This is the novel contribution: TransformerReceiverImpatient

!python -m egg.zoo.channel.train \
    --dir_save=transformer_lazimpa \
    --n_features=100 \
    --vocab_size=40 \
    --max_len=30 \
    --batch_size=32 \
    --batches_per_epoch=1000 \
    --n_epochs=100 \
    --lr=0.001 \
    --probs="powerlaw" \
    --random_seed=42 \
    --sender_cell="transformer" \
    --receiver_cell="transformer" \
    --sender_hidden=512 \
    --receiver_hidden=512 \
    --sender_embedding=256 \
    --receiver_embedding=256 \
    --sender_num_layers=2 \
    --receiver_num_layers=2 \
    --sender_num_heads=8 \
    --receiver_num_heads=8 \
    --causal_sender \
    --causal_receiver \
    --sender_generate_style="in-place" \
    --sender_entropy_coeff=0.1 \
    --receiver_entropy_coeff=0.1 \
    --impatient=1 \
    --reg=1 \
    --length_cost=0.01 \
    --early_stopping_thr=0.9999

### II.3 - Full Paper Replication Settings

Use these settings to replicate the paper results exactly:

In [ ]:
# FULL REPLICATION SETTINGS (uncomment to run)
# WARNING: This takes several hours on GPU!

# TRANSFORMER LAZIMPA - Full Paper Settings
# !python -m egg.zoo.channel.train \
#     --dir_save=transformer_lazimpa_full \
#     --n_features=1000 \
#     --vocab_size=40 \
#     --max_len=30 \
#     --batch_size=32 \
#     --batches_per_epoch=1000 \
#     --n_epochs=500 \
#     --lr=0.001 \
#     --probs="powerlaw" \
#     --random_seed=42 \
#     --sender_cell="transformer" \
#     --receiver_cell="transformer" \
#     --sender_hidden=512 \
#     --receiver_hidden=512 \
#     --sender_embedding=256 \
#     --receiver_embedding=256 \
#     --sender_num_layers=2 \
#     --receiver_num_layers=2 \
#     --sender_num_heads=8 \
#     --receiver_num_heads=8 \
#     --causal_sender \
#     --causal_receiver \
#     --sender_generate_style="in-place" \
#     --sender_entropy_coeff=0.1 \
#     --receiver_entropy_coeff=0.1 \
#     --impatient=1 \
#     --reg=1 \
#     --length_cost=0.01 \
#     --early_stopping_thr=0.9999

## III - Analyze Results

### III.1 - Test Trained Models

In [ ]:
# Test Transformer Baseline
!python -m egg.zoo.channel.test \
    --save_dir="analysis_transformer/" \
    --sender_weights="transformer_baseline/sender/sender_weights100.pth" \
    --receiver_weights="transformer_baseline/receiver/receiver_weights100.pth" \
    --n_features=100 \
    --vocab_size=40 \
    --max_len=30 \
    --sender_cell="transformer" \
    --receiver_cell="transformer" \
    --sender_hidden=512 \
    --receiver_hidden=512 \
    --sender_embedding=256 \
    --receiver_embedding=256 \
    --sender_num_layers=2 \
    --receiver_num_layers=2 \
    --sender_num_heads=8 \
    --receiver_num_heads=8 \
    --causal_sender \
    --causal_receiver \
    --impatient=0

In [ ]:
# Test Transformer LazImpa
!python -m egg.zoo.channel.test \
    --save_dir="analysis_transformer/" \
    --sender_weights="transformer_lazimpa/sender/sender_weights100.pth" \
    --receiver_weights="transformer_lazimpa/receiver/receiver_weights100.pth" \
    --n_features=100 \
    --vocab_size=40 \
    --max_len=30 \
    --sender_cell="transformer" \
    --receiver_cell="transformer" \
    --sender_hidden=512 \
    --receiver_hidden=512 \
    --sender_embedding=256 \
    --receiver_embedding=256 \
    --sender_num_layers=2 \
    --receiver_num_layers=2 \
    --sender_num_heads=8 \
    --receiver_num_heads=8 \
    --causal_sender \
    --causal_receiver \
    --impatient=1

### III.2 - Length Distribution Comparison

In [ ]:
# Compare Transformer Baseline vs LazImpa message lengths
import glob

# Find the last epoch's messages
baseline_files = sorted(glob.glob("transformer_baseline/messages/messages_*.npy"))
lazimpa_files = sorted(glob.glob("transformer_lazimpa/messages/messages_*.npy"))

if baseline_files and lazimpa_files:
    # Load final messages
    baseline_msgs = load_message(baseline_files[-1])
    lazimpa_msgs = load_message(lazimpa_files[-1])
    
    # Get lengths
    baseline_lengths = get_message_lengths(baseline_msgs)
    lazimpa_lengths = get_message_lengths(lazimpa_msgs)
    
    # Plot comparison
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Length by input rank
    axes[0].plot(baseline_lengths, label="Transformer Baseline", alpha=0.8)
    axes[0].plot(lazimpa_lengths, label="Transformer LazImpa", alpha=0.8)
    axes[0].set_xlabel("Input (ranked by frequency)")
    axes[0].set_ylabel("Message Length")
    axes[0].set_title("Message Length by Input Frequency")
    axes[0].legend()
    axes[0].grid(alpha=0.3)
    
    # Histogram
    axes[1].hist(baseline_lengths, bins=20, alpha=0.6, label="Baseline")
    axes[1].hist(lazimpa_lengths, bins=20, alpha=0.6, label="LazImpa")
    axes[1].set_xlabel("Message Length")
    axes[1].set_ylabel("Count")
    axes[1].set_title("Message Length Distribution")
    axes[1].legend()
    axes[1].grid(alpha=0.3)
    
    plt.tight_layout()
    plt.savefig("analysis_transformer/length_comparison.png", dpi=150)
    plt.show()
    
    print(f"\nBaseline - Mean length: {np.mean(baseline_lengths):.2f}, Std: {np.std(baseline_lengths):.2f}")
    print(f"LazImpa  - Mean length: {np.mean(lazimpa_lengths):.2f}, Std: {np.std(lazimpa_lengths):.2f}")
else:
    print("No message files found. Please run training first.")

### III.3 - Accuracy Evolution

In [ ]:
# Compare accuracy evolution
def load_accuracy_evolution(dir_path):
    """Load accuracy values from all epochs."""
    accuracy = []
    acc_files = sorted(glob.glob(f"{dir_path}/accuracy/accuracy_*.npy"))
    for f in acc_files:
        acc = np.load(f)
        accuracy.append(np.mean(acc))
    return accuracy

baseline_acc = load_accuracy_evolution("transformer_baseline")
lazimpa_acc = load_accuracy_evolution("transformer_lazimpa")

if baseline_acc and lazimpa_acc:
    plt.figure(figsize=(10, 5))
    plt.plot(baseline_acc, label="Transformer Baseline", alpha=0.8)
    plt.plot(lazimpa_acc, label="Transformer LazImpa", alpha=0.8)
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.title("Accuracy Evolution - Transformer Models")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.ylim(0, 1)
    plt.savefig("analysis_transformer/accuracy_evolution.png", dpi=150)
    plt.show()
    
    print(f"\nBaseline - Final accuracy: {baseline_acc[-1]:.4f}")
    print(f"LazImpa  - Final accuracy: {lazimpa_acc[-1]:.4f}")
else:
    print("No accuracy files found. Please run training first.")

### III.4 - Mean Length Evolution

In [ ]:
# Compare mean length evolution
def load_length_evolution(dir_path):
    """Load mean message length from all epochs."""
    lengths = []
    msg_files = sorted(glob.glob(f"{dir_path}/messages/messages_*.npy"))
    for f in msg_files:
        msgs = load_message(f)
        lens = get_message_lengths(msgs)
        lengths.append(np.mean(lens))
    return lengths

baseline_len = load_length_evolution("transformer_baseline")
lazimpa_len = load_length_evolution("transformer_lazimpa")

if baseline_len and lazimpa_len:
    plt.figure(figsize=(10, 5))
    plt.plot(baseline_len, label="Transformer Baseline", alpha=0.8)
    plt.plot(lazimpa_len, label="Transformer LazImpa", alpha=0.8)
    plt.xlabel("Epoch")
    plt.ylabel("Mean Message Length")
    plt.title("Mean Length Evolution - Transformer Models")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.savefig("analysis_transformer/length_evolution.png", dpi=150)
    plt.show()
    
    print(f"\nBaseline - Final mean length: {baseline_len[-1]:.2f}")
    print(f"LazImpa  - Final mean length: {lazimpa_len[-1]:.2f}")
else:
    print("No message files found. Please run training first.")

### III.5 - Accuracy vs Mean Length (Trade-off Analysis)

In [ ]:
# Plot accuracy vs mean length trade-off
if baseline_acc and lazimpa_acc and baseline_len and lazimpa_len:
    fig, ax = plt.subplots(figsize=(8, 8))
    
    # Plot trajectory
    ax.scatter(baseline_len, baseline_acc, c=range(len(baseline_len)), 
               cmap='Blues', s=10, alpha=0.7, label="Baseline")
    ax.scatter(lazimpa_len, lazimpa_acc, c=range(len(lazimpa_len)), 
               cmap='Oranges', s=10, alpha=0.7, label="LazImpa")
    
    # Mark start and end
    ax.scatter([baseline_len[0]], [baseline_acc[0]], c='blue', s=100, marker='o', edgecolors='black', zorder=5)
    ax.scatter([baseline_len[-1]], [baseline_acc[-1]], c='blue', s=100, marker='*', edgecolors='black', zorder=5)
    ax.scatter([lazimpa_len[0]], [lazimpa_acc[0]], c='orange', s=100, marker='o', edgecolors='black', zorder=5)
    ax.scatter([lazimpa_len[-1]], [lazimpa_acc[-1]], c='orange', s=100, marker='*', edgecolors='black', zorder=5)
    
    ax.set_xlabel("Mean Message Length")
    ax.set_ylabel("Accuracy")
    ax.set_title("Accuracy vs Mean Length Trade-off\n(circle=start, star=end)")
    ax.legend()
    ax.grid(alpha=0.3)
    ax.set_xlim(0, 32)
    ax.set_ylim(0, 1)
    
    plt.savefig("analysis_transformer/accuracy_vs_length.png", dpi=150)
    plt.show()
else:
    print("No data found. Please run training first.")

### III.6 - Position Analysis (Informative Symbols)

In [ ]:
# Run position analysis for Transformer LazImpa
!python -m egg.zoo.channel.position_analysis \
    --save_dir="analysis_transformer/" \
    --sender_weights="transformer_lazimpa/sender/sender_weights100.pth" \
    --receiver_weights="transformer_lazimpa/receiver/receiver_weights100.pth" \
    --n_features=100 \
    --vocab_size=40 \
    --max_len=30 \
    --sender_cell="transformer" \
    --receiver_cell="transformer" \
    --sender_hidden=512 \
    --receiver_hidden=512 \
    --sender_embedding=256 \
    --receiver_embedding=256 \
    --sender_num_layers=2 \
    --receiver_num_layers=2 \
    --sender_num_heads=8 \
    --receiver_num_heads=8 \
    --causal_sender \
    --causal_receiver \
    --impatient=1

In [ ]:
# Plot position sieve (informative symbols)
import os

sieve_file = "analysis_transformer/position_sieve.npy"
if os.path.exists(sieve_file):
    position_sieve = np.load(sieve_file)
    
    plt.figure(figsize=(12, 8))
    plt.imshow(position_sieve, aspect='auto', cmap='viridis')
    plt.colorbar(label="1=informative, 0=uninformative, -1=no symbol")
    plt.xlabel("Position in Message")
    plt.ylabel("Message (ranked by input frequency)")
    plt.title("Position of Informative Symbols - Transformer LazImpa")
    plt.savefig("analysis_transformer/position_sieve.png", dpi=150)
    plt.show()
else:
    print("Position sieve file not found. Run position_analysis first.")

## IV - Compare LSTM vs Transformer (Optional)

If you have both LSTM and Transformer results, run this section to compare them.

In [ ]:
# Compare all 4 experiment types
experiments = {
    'LSTM Baseline': 'lstm_baseline',
    'LSTM LazImpa': 'lstm_lazimpa', 
    'Transformer Baseline': 'transformer_baseline',
    'Transformer LazImpa': 'transformer_lazimpa'
}

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Load and plot each experiment
for idx, (name, dir_name) in enumerate(experiments.items()):
    ax = axes[idx // 2, idx % 2]
    
    msg_files = sorted(glob.glob(f"{dir_name}/messages/messages_*.npy"))
    if msg_files:
        msgs = load_message(msg_files[-1])
        lengths = get_message_lengths(msgs)
        
        ax.plot(lengths, alpha=0.8)
        ax.axhline(y=np.mean(lengths), color='r', linestyle='--', 
                   label=f'Mean: {np.mean(lengths):.2f}')
        ax.set_title(name)
        ax.set_xlabel("Input (ranked by frequency)")
        ax.set_ylabel("Message Length")
        ax.set_ylim(0, 32)
        ax.legend()
        ax.grid(alpha=0.3)
    else:
        ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes)
        ax.set_title(name)

plt.tight_layout()
plt.savefig("analysis_transformer/all_experiments_comparison.png", dpi=150)
plt.show()

## V - Summary Statistics

In [ ]:
# Print summary statistics for all experiments
print("=" * 60)
print("SUMMARY STATISTICS")
print("=" * 60)

for name, dir_name in experiments.items():
    msg_files = sorted(glob.glob(f"{dir_name}/messages/messages_*.npy"))
    acc_files = sorted(glob.glob(f"{dir_name}/accuracy/accuracy_*.npy"))
    
    if msg_files and acc_files:
        msgs = load_message(msg_files[-1])
        lengths = get_message_lengths(msgs)
        acc = np.mean(np.load(acc_files[-1]))
        
        print(f"\n{name}:")
        print(f"  Final Accuracy: {acc:.4f}")
        print(f"  Mean Length: {np.mean(lengths):.2f}")
        print(f"  Std Length: {np.std(lengths):.2f}")
        print(f"  Min Length: {np.min(lengths)}")
        print(f"  Max Length: {np.max(lengths)}")
    else:
        print(f"\n{name}: No data available")

print("\n" + "=" * 60)